### 1: Inspeção dos Schemas e Localização das Tabelas

In [0]:
# Verificar as tabelas disponíveis no catálogo
display(spark.sql("SHOW TABLES IN workspace.default"))

In [0]:
# Inspecionar as colunas e as primeiras linhas das tabelas Silver de pedidos e itens
print("=== SILVER ECOMMERCE PEDIDOS ===")
df_pedidos = spark.table("workspace.default.silver_ecommerce_pedidos")
print(df_pedidos.columns)
display(df_pedidos.limit(5))

print("=== SILVER ECOMMERCE ITENS PEDIDO ===")
df_itens = spark.table("workspace.default.silver_ecommerce_itens_pedido")
print(df_itens.columns)
display(df_itens.limit(5))

In [0]:
from pyspark.sql import functions as F

# 1. Ler a tabela silver de pedidos
df_pedidos = spark.table("workspace.default.silver_ecommerce_pedidos")

# 2. Extrair data, hora e dia da semana do timestamp do pedido
df_tratado = df_pedidos.withColumn("data_hora", F.to_timestamp("dt_pedido")) \
    .withColumn("data_dia", F.to_date("data_hora")) \
    .withColumn("hora", F.hour("data_hora")) \
    .withColumn("dia_da_semana", F.dayofweek("data_hora")) # 1 = Domingo, 7 = Sábado

# 3. Agregar por dia e hora (Calculando Receita Total e Número de Pedidos)
df_horario = df_tratado.groupBy("data_dia", "hora", "dia_da_semana") \
    .agg(
        F.sum("valor_total").alias("receita"),
        F.count("id_pedido").alias("num_pedidos")
    ) \
    .orderBy("data_dia", "hora")

print("=== AMOSTRA DA SÉRIE TEMPORAL HORÁRIA AGREGADA ===")
display(df_horario.limit(24))

# 4. Salvar a tabela intermediária/gold para uso no baseline e no SARIMAX
df_horario.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_series_temporais_pedidos_hora")

print("Tabela 'gold_series_temporais_pedidos_hora' salva com sucesso na camada Gold!")